# Import data clean

In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/processed/telco_clean.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

# Feature engineering 

##  1. Selisih total aktual vs perkiraan (indikasi perubahan harga)

In [4]:
df['charge_diff'] = df['TotalCharges'] - df['tenure'] * df['MonthlyCharges']
df['charge_diff'].head()

0     0.00
1   -46.80
2     0.45
3   -62.75
4    10.25
Name: charge_diff, dtype: float64

## 2. Jumlah layanan proteksi dan streaming

In [5]:
proteksi  = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
streaming = ['StreamingTV', 'StreamingMovies']
df['n_protection'] = (df[proteksi] == 'Yes').sum(axis=1)
df['n_streaming'] = (df[streaming] == 'Yes').sum(axis=1)

df[['n_protection', 'n_streaming']].head()

,n_protection,n_streaming
0,1,0
1,2,0
2,2,0
3,3,0
4,0,0


## Kelompok Lama berlangganan

In [6]:
df["tenure_bin"] = pd.cut(df["tenure"], bins=[-1, 12, 24, 48, np.inf],
                          labels=["0-12", "13-24", "25-48", "49+"]).astype(str)
df['tenure_bin'].head()

0     0-12
1    25-48
2     0-12
3    25-48
4     0-12
Name: tenure_bin, dtype: str

## Bayar pakai electronic check

In [7]:
df["is_echeck"] = (df["PaymentMethod"] == "Electronic check").astype(int)
df['is_echeck'].head()

0    1
1    0
2    0
3    0
4    1
Name: is_echeck, dtype: int64

## 5. Kelompok yang Punya pasangan atau tanggunga 

In [8]:
df["has_family"] = ((df["Partner"] == "Yes") | (df["Dependents"] == "Yes")).astype(int)
df['has_family'].head()

0    1
1    0
2    0
3    0
4    0
Name: has_family, dtype: int64

# Splitting Dataset

In [9]:
fitur_phase2 = ["Contract", "tenure_bin", "InternetService", "MonthlyCharges", "charge_diff","n_protection",      
                "n_streaming","is_echeck","has_family", "PaperlessBilling"
                ]

X = df[fitur_phase2]
y = df["Churn"].map({"No": 0, "Yes": 1})
print("Jumlah fitur:", X.shape[1])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

Jumlah fitur: 10


((5634, 10), (1409, 10), (5634,), (1409,))

In [10]:
print('label train:',(y_train.value_counts(normalize=True) * 100).round(2).astype(str) + '%')
print('label test:',(y_test.value_counts(normalize=True) * 100).round(2).astype(str) + '%')

label train: Churn
0    73.46%
1    26.54%
Name: proportion, dtype: str
label test: Churn
0    73.46%
1    26.54%
Name: proportion, dtype: str


# Simpan data

In [11]:
data_split = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test
}

joblib.dump(data_split, "../data/processed/data_split_FE.pkl")

['../data/processed/data_split_FE.pkl']